# Crypto.Net · 09 — Security practices

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./09-security.id.ipynb)

What you will do: work through the mistakes that lose funds, and see how Crypto.Net helps avoid each one. Covered: secrets kept in memory, keystores, exact amounts, checksummed addresses, replay protection, verifying before you trust, decoding hostile input, and keeping API keys out of logs.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"

In [ ]:
using System.Numerics;
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Evm;
using Crypto.Net.Bitcoin;
using Crypto.Net.Extensions;
"ready"

## 1. Secrets live in `SecureBuffer`, not `string`

A `SecureBuffer` is pinned (the GC never copies it) and zeroed on `Dispose`. Strings are immutable, so they linger in memory and show up in crash dumps. Analyzer rule CNET002 flags secret-looking `string` fields and properties.

In [ ]:
var secret = SecureBuffer.FromHex("4c0883a69102937d6231471b5dbb6204fe5129617082792ae468d01a3f362318");
Console.WriteLine(secret);                         // never prints the bytes
var address = EvmAddress.FromPrivateKey(secret.Span);
secret.Dispose();
Console.WriteLine($"{address} · disposed: {secret.IsDisposed}");
try { _ = secret.Span.Length; } catch (ObjectDisposedException) { Console.WriteLine("reading a disposed secret throws"); }

## 2. Keys at rest go in a keystore

Web3 Secret Storage v3 (scrypt plus AES-128-CTR plus a MAC) is the format geth and MetaMask use. A wrong password fails the MAC; it never yields a wrong key.

In [ ]:
byte[] demoKey = CryptoNative.Keccak256("notebook demo key — never fund it"u8);
string json = KeyStore.Encrypt(demoKey, "correct horse battery staple", kdf: KeyStoreKdf.Light);
var restored = KeyStore.Decrypt(json, "correct horse battery staple");
Console.WriteLine($"round trip ok: {restored.ContentEquals(demoKey)}");
restored.Dispose();
try { KeyStore.Decrypt(json, "wrong"); } catch (System.Security.Cryptography.CryptographicException e) { Console.WriteLine($"wrong password: {e.Message}"); }

## 3. Amounts are exact

`0.1 + 0.2 != 0.3` in binary floating point. `Amount` stores integer base units. It refuses any conversion that would lose precision. Analyzer rule CNET003 flags amounts built from `double`.

In [ ]:
Console.WriteLine($"double: 0.1 + 0.2 = {0.1 + 0.2:R}");
var a = Amount.Parse("0.1", 18) + Amount.Parse("0.2", 18);
Console.WriteLine($"Amount: {a} ({a.BaseUnits} wei)");
try { Amount.Parse("1.0000001", 6); } catch (FormatException e) { Console.WriteLine($"too many decimals for USDC: {e.Message}"); }

## 4. Checksums catch typos

EIP-55 mixed case, Bech32 and Base58Check all detect most single-character mistakes. Validate addresses before you send anything.

In [ ]:
string good = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAed";
string typo = "0x5aAeb6053F3E94C9b9A09f33669435E7Ef1BeAeD";
string btcTypo = "bc1qcr8te4kr609gcawutmrza0j4xv80jy8z306fyv";
new { EvmGood = EvmAddress.IsValid(good), EvmTypo = EvmAddress.IsValid(typo), BitcoinTypo = BitcoinAddress.IsValid(btcTypo) }

## 5. Replay protection and verification

EIP-155 binds a signature to one chain id, so a Sepolia transaction cannot be replayed on mainnet. When someone hands you a signed message, recover the signer and compare addresses. Never trust the address the sender claims.

In [ ]:
var tx = new EvmTransaction { ChainId = EvmChain.Sepolia.NumericChainId, Nonce = 0, To = good, Value = 1, MaxFeePerGas = 1, MaxPriorityFeePerGas = 1 };
var signed = tx.Sign(demoKey);
var decoded = EvmTransaction.DecodeSigned(signed.RawBytes);
Console.WriteLine($"signed for chain {decoded.Transaction.ChainId} (Sepolia), not 1 (mainnet)");

byte[] signature = EvmMessageSigner.SignPersonalMessage(demoKey, "I own this address");
string signer = EvmMessageSigner.RecoverPersonalMessageSigner("I own this address", HexUtil.Encode(signature));
Console.WriteLine($"claimed {EvmAddress.FromPrivateKey(demoKey)} · recovered {signer}");
Console.WriteLine($"tampered message recovers someone else: {EvmMessageSigner.RecoverPersonalMessageSigner("I own this addresS", HexUtil.Encode(signature))}");

## 6. Hostile input fails cleanly

PSBTs, transactions, RLP, SCALE and Solana messages come from untrusted sources. CI fuzzes every decoder; malformed input raises `FormatException`, never a crash. The PSBT signer also refuses inputs whose UTXO data does not match the transaction it spends.

In [ ]:
foreach (string bad in new[] { "cHNidP8BAA==", "70736274ff0100ffffffff", "deadbeef" })
    Console.WriteLine($"{bad,-24} → valid PSBT: {Psbt.TryParse(bad, out _)}");
try { EvmTransaction.DecodeSigned(Convert.FromHexString("f8ff00")); }
catch (FormatException e) { Console.WriteLine($"RLP: {e.Message}"); }

## 7. API keys stay out of logs

Ankr puts the key in the URL path. `RpcEndpoint.ToString()` redacts it, and `AddCryptoNet` removes the HTTP loggers whenever keyed providers are configured.

In [ ]:
var endpoint = RpcProviders.Ankr.GetEndpoint(EvmChain.Sepolia, "my-secret-api-key")!;
$"logged as: {endpoint}"

## Checklist

- Develop on testnets. CNET004 flags mainnet clients in Debug builds.
- Keep phrases and keys in user secrets, a KMS or a hardware signer. CNET001 flags hard-coded ones.
- Simulate before you sign: `PrepareTransactionAsync`, `SimulateTransactionAsync` and `SimulateAsync` all do this.
- Pass `CancellationToken`s, so a stuck RPC call cannot hang your service (CNET005).
- See the [security guide](../../docs/en/security.md).